In [0]:

# ============================================================
# SHIPMENT TRACKING PROJECT
# BRONZE INGESTION USING AUTO LOADER
# ============================================================

from pyspark.sql.functions import *

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# Actual Unity Catalog Volume path

catalog = "logistics"
schema = "bronze"

# Volume path
base_path = "/Volumes/logistics/bronze/volume"

# Source paths
orders_path = base_path + "/transaction_data/shipment_orders/"

events_path = base_path + "/events_data/"

master_path = base_path + "/master_data/"

# Checkpoint location
checkpoint_base = base_path + "/_autoloader"

# Bronze schema
bronze_schema = "logistics.bronze"

spark.sql("""
CREATE SCHEMA IF NOT EXISTS logistics.bronze
""")


# ------------------------------------------------------------
# 2. DEFINE SHIPMENT ORDERS SCHEMA
# ------------------------------------------------------------

orders_schema = """
shipment_id STRING,
order_id STRING,
customer_id STRING,
carrier_id STRING,
origin_warehouse_id STRING,
destination_city STRING,
created_ts STRING,
promised_delivery_ts STRING,
package_weight_kg STRING,
service_level STRING,
shipping_cost STRING,
shipment_status STRING
"""


# ------------------------------------------------------------
# 3. DEFINE TRACKING EVENTS SCHEMA
# ------------------------------------------------------------

tracking_schema = """
tracking_event_id STRING,
shipment_id STRING,
event_ts STRING,
facility_id STRING,
event_type STRING,
event_city STRING,
scan_source STRING,
carrier_id STRING,
scan_quality STRING,
received_ts STRING
"""


# ------------------------------------------------------------
# 4. DEFINE DELIVERY ATTEMPTS SCHEMA
# ------------------------------------------------------------

attempts_schema = """
attempt_id STRING,
shipment_id STRING,
attempt_ts STRING,
driver_id STRING,
attempt_result STRING,
failure_reason STRING,
proof_of_delivery STRING,
delivery_city STRING,
attempt_sequence STRING,
received_ts STRING
"""


# ------------------------------------------------------------
# 5. REUSABLE AUTO LOADER FUNCTION
# ------------------------------------------------------------

def load_to_bronze(
    source_path,
    source_schema,
    table_name,
    checkpoint_name,
    file_filter
):

    schema_path = (
        checkpoint_base + "/" + checkpoint_name + "_schema"
    )

    checkpoint_path = (
        checkpoint_base + "/" + checkpoint_name + "_checkpoint"
    )

    print("Starting ingestion:", table_name)

    # Read CSV files using Auto Loader
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .option("pathGlobFilter", file_filter)
        .option("cloudFiles.schemaHints", source_schema)
        .load(source_path)
    )

    # Add source metadata
    df = (
        df
        .withColumn(
            "_source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "_ingestion_timestamp",
            current_timestamp()
        )
    )

    # Write to Bronze Delta table
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(table_name)
    )

    query.awaitTermination()

    print("Successfully completed:", table_name)


# ------------------------------------------------------------
# 6. INGEST SHIPMENT ORDERS (12 BATCH FILES)
# ------------------------------------------------------------

load_to_bronze(
    orders_path,
    orders_schema,
    bronze_schema + ".shipment_orders",
    "orders",
    "shipment_orders_batch_*.csv"
)


# ------------------------------------------------------------
# 7. INGEST SHIPMENT TRACKING EVENTS
# ------------------------------------------------------------

load_to_bronze(
    events_path,
    tracking_schema,
    bronze_schema + ".shipment_tracking_events",
    "tracking",
    "shipment_tracking_events.csv"
)


# ------------------------------------------------------------
# 8. INGEST DELIVERY ATTEMPTS
# ------------------------------------------------------------

load_to_bronze(
    events_path,
    attempts_schema,
    bronze_schema + ".delivery_attempts",
    "attempts",
    "delivery_attempts.csv"
)


# ------------------------------------------------------------
# 9. VERIFY BRONZE TABLES
# ------------------------------------------------------------

print("SHIPMENT ORDERS")

display(
    spark.table(
        bronze_schema + ".shipment_orders"
    )
)

print("TRACKING EVENTS")

display(
    spark.table(
        bronze_schema + ".shipment_tracking_events"
    )
)

print("DELIVERY ATTEMPTS")

display(
    spark.table(
        bronze_schema + ".delivery_attempts"
    )
)


# ------------------------------------------------------------
# 10. CHECK RECORD COUNTS
# ------------------------------------------------------------

for table in [
    "shipment_orders",
    "shipment_tracking_events",
    "delivery_attempts"
]:

    count=spark.table(bronze_schema + "." + table).count()

    print(table, ":", count)
